# Week 5 Pair Programming: Cluster Detective 🔍

**Duration:** 20 minutes  
**Scaffolding:** 80% (code provided, focus on interpretation)  
**Dataset:** Wine (178 samples, 13 chemical properties)  

---

## Your Mission

You're a data scientist for a wine distributor. Your boss says:

> *"We have 178 wines with chemical analysis data. Find natural groupings so we can segment our inventory. What types of wine do we have?"*

**Your job:**
1. Run K-Means clustering (code provided)
2. Analyze cluster characteristics
3. **NAME each cluster** based on chemical properties
4. Decide: Are these meaningful wine categories?

**Remember:** No "correct" names! It's about **interpretation**.

---

## Part 1: Load & Prepare Data (Code Provided)

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load wine dataset
wine = load_wine()
X = wine.data
feature_names = wine.feature_names

print(f"Dataset shape: {X.shape}")
print(f"\nFeatures: {feature_names}")

In [ ]:
# Create DataFrame for easier analysis
df = pd.DataFrame(X, columns=feature_names)
df.head()

In [ ]:
# Check feature statistics
df.describe()

In [ ]:
# Scale features (CRITICAL for K-Means!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Features scaled (mean=0, std=1)")

---

## Part 2: Determine Optimal K (Code Provided)

Let's use the elbow method and silhouette scores to guide our choice of K.

In [ ]:
# Elbow Method
inertias = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from yellowbrick.cluster import KElbowVisualizer
model = KMeans()
visualizer = KElbowVisualizer(model, k=(1,11))
visualizer.fit(X)
visualizer.show()

In [ ]:
# Silhouette Scores
silhouette_scores = []
K_range_sil = range(2, 11)  # Silhouette needs at least 2 clusters

for k in K_range_sil:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k}: Silhouette Score = {score:.3f}")

# Plot
plt.figure(figsize=(8, 5))
plt.plot(K_range_sil, silhouette_scores, marker='o', color='orange')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis for Optimal K')
plt.axhline(y=0.5, color='gray', linestyle='--', label='Good threshold (0.5)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer
model = SilhouetteVisualizer(KMeans(n_clusters=3, random_state=42))
model.fit(X_scaled)
model.show()

---

## 🧩 TASK 1: Choose K (YOUR TURN!)

**Discuss with your partner:**

1. Where is the "elbow" in the first plot?
2. Which K has the highest silhouette score?
3. What K would you choose? Why?

**Write your answer below:**

**YOUR ANSWER:**

- **Elbow at K =** ___ (fill in)
- **Best silhouette K =** ___ (fill in)
- **Our choice: K =** ___ because:
  - (Write 1-2 sentences explaining your reasoning)

---

## Part 3: Fit K-Means with Chosen K

**UPDATE:** Change `n_clusters=3` to your chosen K value below!

In [ ]:
# Fit K-Means with your chosen K
K_chosen = 3  # ⬅️ UPDATE THIS to your chosen K

kmeans = KMeans(n_clusters=K_chosen, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels to DataFrame
df['cluster'] = clusters

print(f"✅ K-Means fitted with K={K_chosen}")
print(f"\nCluster distribution:")
print(df['cluster'].value_counts().sort_index())

---

## Part 4: Analyze Cluster Characteristics (Code Provided)

Let's see what defines each cluster!

In [ ]:
# Calculate mean values for each cluster
cluster_means = df.groupby('cluster').mean()
cluster_means

In [ ]:
# Visualize cluster characteristics (heatmap)
plt.figure(figsize=(12, 4))
sns.heatmap(cluster_means.T, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Cluster Characteristics (Mean Values)')
plt.xlabel('Cluster')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Focus on key distinguishing features
key_features = ['alcohol', 'color_intensity', 'proline', 'flavanoids']

cluster_means[key_features].plot(kind='bar', figsize=(10, 5))
plt.title('Key Features by Cluster')
plt.xlabel('Cluster')
plt.ylabel('Mean Value (scaled)')
plt.legend(title='Feature', bbox_to_anchor=(1.05, 1))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---

## 🧩 TASK 2: Name Your Clusters! (YOUR TURN!)

**This is the MAIN task!** Look at the heatmap and bar charts above.

**For each cluster, answer:**
1. What are its **defining characteristics**? (High/low in which features?)
2. What **name** would you give this cluster?
3. How would you **describe** this wine type to a customer?

**Fill in the table below:**

### Cluster 0

**Defining characteristics:**
- (e.g., "High in alcohol, low in color_intensity")
- (List 2-3 key features)

**Cluster name:** ___ (e.g., "Full-bodied Reds", "Light Whites", etc.)

**Customer description:**
- (Write 1-2 sentences describing this wine type)

---

### Cluster 1

**Defining characteristics:**
- 
- 

**Cluster name:** ___

**Customer description:**
- 

---

### Cluster 2

**Defining characteristics:**
- 
- 

**Cluster name:** ___

**Customer description:**
- 

---

*(If you chose K>3, add more cluster interpretations above)*

---

## Part 5: Visualize Clusters (Code Provided)

Since we have 13 dimensions, let's visualize the first two principal components for interpretation.

In [ ]:
from sklearn.decomposition import PCA

# Apply PCA for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Create visualization
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Wine Clusters Visualized (PCA 2D Projection)')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total variance explained by PC1 + PC2: {pca.explained_variance_ratio_.sum():.1%}")

---

## 🧩 TASK 3: Final Evaluation (YOUR TURN!)

**Discuss with your partner and answer:**

**1. Do the clusters look well-separated in the PCA plot?**
- (Yes/No and why)

**2. Are these meaningful wine categories for a distributor?**
- (Yes/No and explain)

**3. What would you recommend to your boss?**
- (e.g., "Use 3 categories", "Need domain expert to validate", etc.)

**4. If you had wine expert knowledge, what would you want to know about these clusters?**
- (e.g., "Grape variety", "Region", "Price point", etc.)

---

## 🎯 Key Takeaways

**From this exercise, you learned:**

1. ✅ **Choosing K:** Combine elbow + silhouette + domain knowledge
2. ✅ **Interpretation is key:** No "correct" cluster names - it's about making sense of patterns
3. ✅ **Feature scaling matters:** K-Means uses distances
4. ✅ **Domain knowledge crucial:** Chemical properties ≠ business categories without expert input
5. ✅ **PCA for visualization:** Project high-D clusters to 2D for inspection

**The big idea:**
> *Unsupervised learning finds patterns. HUMANS interpret meaning.*

---

## Bonus Challenge (If Time Permits)

**Try K=2 or K=4 and see how interpretations change!**

- Does K=2 oversimplify?
- Does K=4 overcomplicate?
- Which K tells the best story?